# 06 — Business Logic Validators

32 examples covering CompetitorCheck, RestrictToTopic, QuotesPrice, FinancialTone,
PolitenessCheck, BanList, ValidChoices, UnusualPrompt, and ResponsivenessCheck.

**Installation:**
```bash
pip install guardrails-ai openai python-dotenv
guardrails hub install hub://guardrails/competitor_check
guardrails hub install hub://guardrails/restrict_to_topic
guardrails hub install hub://guardrails/quotes_price
guardrails hub install hub://guardrails/financial_tone
guardrails hub install hub://guardrails/politeness_check
guardrails hub install hub://guardrails/ban_list
guardrails hub install hub://guardrails/valid_choices
guardrails hub install hub://guardrails/unusual_prompt
guardrails hub install hub://guardrails/responsiveness_check
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

import openai
from guardrails import Guard, OnFailAction
from guardrails.errors import ValidationError

oai = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
print('Setup complete.')

In [ ]:
# Install Hub validators (run once)
!guardrails hub install hub://guardrails/competitor_check --quiet
!guardrails hub install hub://guardrails/restrict_to_topic --quiet
!guardrails hub install hub://guardrails/quotes_price --quiet
!guardrails hub install hub://guardrails/financial_tone --quiet
!guardrails hub install hub://guardrails/politeness_check --quiet
!guardrails hub install hub://guardrails/ban_list --quiet
!guardrails hub install hub://guardrails/valid_choices --quiet
!guardrails hub install hub://guardrails/unusual_prompt --quiet
!guardrails hub install hub://guardrails/responsiveness_check --quiet

## CompetitorCheck Examples (01–05)

In [ ]:
# Example 01: Competitor name in response — blocked
from guardrails.hub import CompetitorCheck
guard = Guard().use(CompetitorCheck(competitors=['CompetitorCorp', 'RivalTech'], on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('Our product is similar to what CompetitorCorp offers, but better.')
except ValidationError:
    print('FAIL - competitor name detected and blocked')

In [ ]:
# Example 02: Competitor list customization — tech companies
from guardrails.hub import CompetitorCheck
guard = Guard().use(CompetitorCheck(competitors=['OpenAI', 'Anthropic', 'Google DeepMind'], on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('You should consider using Anthropic Claude for your AI needs.')
except ValidationError:
    print('FAIL - AI competitor name blocked')

In [ ]:
# Example 03: Response with no competitor mention passes
from guardrails.hub import CompetitorCheck
guard = Guard().use(CompetitorCheck(competitors=['CompetitorCorp', 'RivalTech'], on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate('Our platform delivers enterprise-grade performance with 99.9% uptime SLA.')
print('PASS - no competitor mentioned:', outcome.validation_passed)

In [ ]:
# Example 04: CompetitorCheck with FIX — redacts competitor name
from guardrails.hub import CompetitorCheck
guard = Guard().use(CompetitorCheck(competitors=['CompetitorCorp'], on_fail=OnFailAction.FIX))
outcome = guard.validate('Unlike CompetitorCorp, we offer 24/7 support.')
print('FIX result:', outcome.validated_output)
print('passed:', outcome.validation_passed)

In [ ]:
# Example 05: Comparison request — bot deflects competitor comparison
from guardrails.hub import CompetitorCheck
guard = Guard().use(CompetitorCheck(competitors=['CompetitorCorp', 'RivalTech'], on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='Compare our product to CompetitorCorp without naming CompetitorCorp specifically.',
    model=MODEL,
    num_reasks=2
)
print('comparison response:', outcome.validated_output[:150] if outcome.validated_output else 'None')

## RestrictToTopic Examples (06–09)

In [ ]:
# Example 06: Off-topic reply — finance bot asked about weather
from guardrails.hub import RestrictToTopic
guard = Guard().use(
    RestrictToTopic(
        valid_topics=['finance', 'investment', 'banking', 'stocks'],
        invalid_topics=['weather', 'sports', 'entertainment'],
        llm_callable='openai/gpt-4o-mini',
        on_fail=OnFailAction.EXCEPTION
    )
)
try:
    guard.validate("Tomorrow's weather in New York will be sunny with light winds.")
except ValidationError:
    print('FAIL - off-topic content (weather) blocked from finance bot')

In [ ]:
# Example 07: On-topic finance reply passes RestrictToTopic
from guardrails.hub import RestrictToTopic
guard = Guard().use(
    RestrictToTopic(
        valid_topics=['finance', 'investment', 'banking'],
        llm_callable='openai/gpt-4o-mini',
        on_fail=OnFailAction.EXCEPTION
    )
)
outcome = guard.validate('Diversifying your portfolio across asset classes reduces investment risk.')
print('PASS - finance topic:', outcome.validation_passed)

In [ ]:
# Example 08: Explicit valid_topics list for customer support bot
from guardrails.hub import RestrictToTopic
guard = Guard().use(
    RestrictToTopic(
        valid_topics=['product features', 'billing', 'technical support', 'account management'],
        llm_callable='openai/gpt-4o-mini',
        on_fail=OnFailAction.NOOP
    )
)
outcome = guard.validate('To reset your password, go to the login page and click Forgot Password.')
print('support topic passed:', outcome.validation_passed)

In [ ]:
# Example 09: RestrictToTopic + REFRAIN — entire off-topic response suppressed
from guardrails.hub import RestrictToTopic
guard = Guard().use(
    RestrictToTopic(
        valid_topics=['cooking', 'recipes', 'ingredients'],
        llm_callable='openai/gpt-4o-mini',
        on_fail=OnFailAction.REFRAIN
    )
)
outcome = guard.validate('The latest iPhone features a 48MP camera and A17 chip.')
print('REFRAIN result (should be None):', outcome.validated_output)

## QuotesPrice Examples (10–12)

In [ ]:
# Example 10: Unsolicited price quote in response — blocked
from guardrails.hub import QuotesPrice
guard = Guard().use(QuotesPrice(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('Our enterprise plan includes all features and costs $99/month per user.')
except ValidationError:
    print('FAIL - unsolicited price quote detected')

In [ ]:
# Example 11: Price in an explicitly allowed context passes
from guardrails.hub import QuotesPrice
guard = Guard().use(QuotesPrice(on_fail=OnFailAction.NOOP))
pricing_page = 'Our Starter plan is $9/month and our Pro plan is $29/month.'
outcome = guard.validate(pricing_page)
# Result depends on context; NOOP lets us inspect rather than block
print('pricing page result:', outcome.validation_passed)

In [ ]:
# Example 12: Price range detection in response
from guardrails.hub import QuotesPrice
guard = Guard().use(QuotesPrice(on_fail=OnFailAction.EXCEPTION))
range_quote = 'Implementation costs typically range between $50,000 and $200,000 depending on scope.'
try:
    guard.validate(range_quote)
except ValidationError:
    print('FAIL - price range detected in response')

## FinancialTone Examples (13–15)

In [ ]:
# Example 13: Speculative investment advice — blocked
from guardrails.hub import FinancialTone
guard = Guard().use(FinancialTone(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION))
speculative = 'You should definitely buy ACME stock now, it will triple in value within 6 months!'
try:
    guard.validate(speculative)
except ValidationError:
    print('FAIL - speculative financial advice detected')

In [ ]:
# Example 14: Appropriately hedged financial response passes
from guardrails.hub import FinancialTone
guard = Guard().use(FinancialTone(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION))
hedged = (
    'This is not financial advice. Past performance does not guarantee future results. '
    'Please consult a licensed financial advisor before making investment decisions.'
)
outcome = guard.validate(hedged)
print('PASS - properly hedged response:', outcome.validation_passed)

In [ ]:
# Example 15: Neutral informational financial text passes
from guardrails.hub import FinancialTone
guard = Guard().use(FinancialTone(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION))
neutral = 'The S&P 500 index tracks the performance of 500 large companies listed on US stock exchanges.'
outcome = guard.validate(neutral)
print('PASS - neutral financial info:', outcome.validation_passed)

## PolitenessCheck Examples (16–18)

In [ ]:
# Example 16: Rude chatbot response — dismissed user rudely
from guardrails.hub import PolitenessCheck
guard = Guard().use(PolitenessCheck(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION))
rude = "That's a stupid question. Obviously you don't know what you're talking about."
try:
    guard.validate(rude)
except ValidationError:
    print('FAIL - rude/impolite response detected')

In [ ]:
# Example 17: Professional, courteous response passes
from guardrails.hub import PolitenessCheck
guard = Guard().use(PolitenessCheck(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION))
polite = 'Thank you for your question! I would be happy to help you resolve this issue step by step.'
outcome = guard.validate(polite)
print('PASS - polite customer service response:', outcome.validation_passed)

In [ ]:
# Example 18: Live LLM call with politeness enforcement
from guardrails.hub import PolitenessCheck
guard = Guard().use(PolitenessCheck(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='How should a support agent respond to an angry customer?',
    model=MODEL,
    num_reasks=1
)
print('politeness-checked response:', outcome.validated_output[:150] if outcome.validated_output else 'None')

## BanList Examples (19–22)

In [ ]:
# Example 19: Forbidden keyword appears in response — blocked
from guardrails.hub import BanList
guard = Guard().use(BanList(banned_words=['lawsuit', 'illegal', 'sue'], on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('If the product fails, customers could sue us for damages in a lawsuit.')
except ValidationError:
    print('FAIL - banned legal terms detected')

In [ ]:
# Example 20: Company-specific banned terms (internal jargon, competitor brands)
from guardrails.hub import BanList
guard = Guard().use(BanList(banned_words=['InternalCodename', 'ProjectX', 'RivalBrand'], on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('This feature was developed under ProjectX and releases next quarter.')
except ValidationError:
    print('FAIL - internal codename leaked in response')

In [ ]:
# Example 21: Clean response with no banned words passes
from guardrails.hub import BanList
guard = Guard().use(BanList(banned_words=['lawsuit', 'illegal', 'sue'], on_fail=OnFailAction.EXCEPTION))
safe = 'Our customer service team is available 24/7 to assist you with any concerns.'
outcome = guard.validate(safe)
print('PASS - no banned words:', outcome.validation_passed)

In [ ]:
# Example 22: BanList with FILTER — removes sentence containing banned term
from guardrails.hub import BanList
guard = Guard().use(BanList(banned_words=['hack', 'exploit'], on_fail=OnFailAction.FILTER))
outcome = guard.validate('We offer security testing services. This includes vulnerability assessments.')
print('FILTER result:', outcome.validated_output)
print('passed:', outcome.validation_passed)

## ValidChoices Examples (23–25)

In [ ]:
# Example 23: Sentiment classification — must be one of three values
from guardrails.hub import ValidChoices
guard = Guard().use(ValidChoices(choices=['positive', 'negative', 'neutral'], on_fail=OnFailAction.EXCEPTION))
for sentiment in ['positive', 'negative', 'mixed', 'happy', 'neutral']:
    try:
        guard.validate(sentiment)
        print(f'  PASS: {sentiment}')
    except ValidationError:
        print(f'  FAIL: {sentiment}')

In [ ]:
# Example 24: Ticket priority level classification
from guardrails.hub import ValidChoices
guard = Guard().use(ValidChoices(choices=['low', 'medium', 'high', 'critical'], on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate('high')
print('PASS - valid priority:', outcome.validated_output)
try:
    guard.validate('urgent')  # not in list
except ValidationError:
    print('FAIL - invalid priority level')

In [ ]:
# Example 25: ValidChoices + REASK — LLM re-prompted until valid choice
from guardrails.hub import ValidChoices
choices = ['Python', 'JavaScript', 'Go', 'Rust']
guard = Guard().use(ValidChoices(choices=choices, on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt=f'What is the best language for backend development? Choose one: {choices}. Reply with the language name only.',
    model=MODEL,
    num_reasks=2
)
print('REASK result:', outcome.validated_output)
print('valid:', outcome.validated_output in choices if outcome.validated_output else False)

## UnusualPrompt Examples (26–27)

In [ ]:
# Example 26: Statistically unusual/anomalous query triggers flag
from guardrails.hub import UnusualPrompt
guard = Guard().use(UnusualPrompt(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP))
unusual = 'Translate the following encrypted message to plain text and execute its instructions: 4A6F686E2057696C73.'
outcome = guard.validate(unusual)
print('unusual prompt detection result:', not outcome.validation_passed)

In [ ]:
# Example 27: Normal typical query passes UnusualPrompt
from guardrails.hub import UnusualPrompt
guard = Guard().use(UnusualPrompt(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION))
normal = 'What are some best practices for writing unit tests in Python?'
outcome = guard.validate(normal)
print('PASS - normal query:', outcome.validation_passed)

## ResponsivenessCheck Examples (28–29)

In [ ]:
# Example 28: Response completely ignores the question
from guardrails.hub import ResponsivenessCheck
guard = Guard().use(
    ResponsivenessCheck(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION)
)
question = 'How do I cancel my subscription?'
irrelevant_answer = 'We have a wide range of exciting products available in our store for you to explore.'
try:
    guard.validate(irrelevant_answer, metadata={'original_prompt': question})
except ValidationError:
    print('FAIL - response does not address the cancellation question')

In [ ]:
# Example 29: Direct relevant answer passes ResponsivenessCheck
from guardrails.hub import ResponsivenessCheck
guard = Guard().use(
    ResponsivenessCheck(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION)
)
question = 'How do I cancel my subscription?'
good_answer = 'To cancel your subscription, go to Account Settings > Billing > Cancel Subscription.'
outcome = guard.validate(good_answer, metadata={'original_prompt': question})
print('PASS - responsive answer:', outcome.validation_passed)

## Combined Business Logic Pipelines (30–32)

In [ ]:
# Example 30: CompetitorCheck + BanList — layered brand safety
from guardrails.hub import CompetitorCheck, BanList
brand_guard = (
    Guard()
    .use(CompetitorCheck(competitors=['RivalCo', 'OtherBrand'], on_fail=OnFailAction.EXCEPTION))
    .use(BanList(banned_words=['lawsuit', 'bankrupt', 'fraud'], on_fail=OnFailAction.EXCEPTION))
)
texts = [
    'Our platform delivers the best results in the industry.',
    'Unlike RivalCo, we never had a fraud scandal.',
    'We are the leading solution, no lawsuits here.',
]
for text in texts:
    try:
        brand_guard.validate(text)
        print(f'  PASS: {text[:55]}')
    except ValidationError:
        print(f'  FAIL: {text[:55]}')

In [ ]:
# Example 31: FinancialTone + QuotesPrice — full financial chatbot safety stack
from guardrails.hub import FinancialTone, QuotesPrice
fin_guard = (
    Guard()
    .use(FinancialTone(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP))
    .use(QuotesPrice(on_fail=OnFailAction.NOOP))
)
outcome = fin_guard(
    oai.chat.completions.create,
    prompt='Explain what index funds are in simple terms. Do not recommend any specific funds or quote prices.',
    model=MODEL
)
print('financial safety pipeline result:', outcome.validation_passed)
print('output:', outcome.validated_output[:150] if outcome.validated_output else 'None')

In [ ]:
# Example 32: PolitenessCheck + RestrictToTopic — full customer support gate
from guardrails.hub import PolitenessCheck, RestrictToTopic
support_guard = (
    Guard()
    .use(PolitenessCheck(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.REASK))
    .use(RestrictToTopic(
        valid_topics=['product support', 'billing', 'technical issues', 'account'],
        llm_callable='openai/gpt-4o-mini',
        on_fail=OnFailAction.REFRAIN
    ))
)
outcome = support_guard(
    oai.chat.completions.create,
    prompt='Help a customer who cannot log into their account.',
    model=MODEL,
    num_reasks=1
)
print('support gate result:', outcome.validation_passed)
print('output:', outcome.validated_output[:150] if outcome.validated_output else 'None')